In [0]:
# COMMAND ----------

import pandas as pd
import os

# 1. RUTA: Apuntando a la carpeta 'Datos' del repositorio clonado en tu Workspace
folder_path = "/Workspace/Users/23christian.nina23@gmail.com/Proyecto-SQL-Caso_Estudio_Kiosko/Datos"

# 2. ESQUEMA: Destino en proyecto_kiosko
catalog_schema = "cursodataacademy.proyecto_kiosko"

# Buscamos los archivos CSV en la carpeta del repositorio
csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]

print(f"Se encontraron {len(csv_files)} archivos CSV:\n")

for csv_file in sorted(csv_files):
    file_path = os.path.join(folder_path, csv_file)
    table_name = csv_file.replace(".csv", "").lower()
    full_table_name = f"{catalog_schema}.{table_name}"

    # 3. MANEJO DE SEPARADORES: Tickets usan ';' y Productos usan ','
    separador = ";" if "ticket" in csv_file.lower() else ","

    # Leer CSV con el separador dinámico
    df_pandas = pd.read_csv(file_path, encoding="utf-8-sig", sep=separador, skipinitialspace=True)

    # Reemplaza los espacios y caracteres problemáticos por guiones bajos
    df_pandas.columns = df_pandas.columns.str.replace(r'[ ,;{}()\n\t=]', '_', regex=True)

    # Crear DataFrame de Spark y guardar como tabla Delta
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)

    row_count = df_spark.count()
    col_count = len(df_spark.columns)
    print(f"✅ {csv_file} → {full_table_name} ({row_count} filas, {col_count} columnas: {df_spark.columns})")

print(f"\n✨ Carga completa: {len(csv_files)} tablas creadas en {catalog_schema}")